# 06 - Sentence Embeddings & Semantic Similarity

**Goal:** find feedback messages that are *semantically* similar — even when they use different words.

> "Payment failed during checkout."
> "Unable to complete my card transaction."
>
> These should be similar (both payment problems), even though they share almost no words.

**Approach:** `sentence-transformers` `all-MiniLM-L6-v2` maps each sentence to a 384-dim vector. Cosine similarity of the vectors measures semantic closeness.

> **Requirement:** this notebook needs `sentence-transformers` (and `torch`). If missing, install:
> ```
> pip install sentence-transformers
> ```
> The first cell downloads the model weights (~80 MB).

In [ ]:
import sys
sys.path.insert(0, "..")

from src.embeddings import EmbeddingEngine

engine = EmbeddingEngine()
print("Loaded model:", engine.model_name)

## Cosine similarity demo

Two sentences about the same (payment) problem should score high, while unrelated ones should score low.

In [ ]:
pairs = [
    ("Payment failed during checkout.", "Unable to complete my card transaction."),
    ("Payment failed during checkout.", "The app is very slow."),
    ("I cannot log in.", "Login is broken for me."),
    ("I love the new design.", "The new UI is great."),
]

for a, b in pairs:
    sim = engine.compute_similarity(a, b)
    print(f"{sim:.3f}  | {a[:40]}  <->  {b[:40]}")

The first pair (payment failure) should score much higher than the second. This is the power of learned embeddings: they capture meaning, not just shared words.

## Similar feedback search

Given one feedback message, return the top-3 most semantically similar messages from a corpus.

In [ ]:
import pandas as pd
from src.data_loader import load_raw_tweets

# Use a small sample so indexing is fast
df = pd.DataFrame({"feedback": load_raw_tweets()["text"].astype(str)})
sample = df.sample(500, random_state=42)["feedback"].tolist()

engine.build_index(sample, show_progress=False)
print("Index built over", len(sample), "feedback messages")

In [ ]:
query = "my payment was declined and I was double charged"
print(f"QUERY: {query}\n")
for r in engine.find_similar(query, top_k=3):
    print(f"  {r['similarity']:.3f} | {r['text'][:100]}")

## Application

Sentence embeddings power:
- **Similar feedback retrieval** (shown above)
- **Clustering** feedback into themes
- **Deduplication** of similar complaints
- Semantic search in support ticketing

## Limitation

Embeddings are great for similarity but are NOT directly interpretable (we can't easily say *why* two texts are similar). For that, the TF-IDF keyword approach in notebook 05 is better.